# Poke Agent Training

This notebook runs the **temporal transformer** training pipeline via the `poke_agent` package.

- **Edit settings in** `poke_agent/config.py` (epochs, batch size, model size, data paths).
- **Hand tracking:** each game step parses CABT `logs` to infer draw timing, hand age, and opponent hidden-hand signals. Windows and trackers **reset every episode** — no memory across games.
- **Outputs:** checkpoint → `outputs/checkpoints/temporal_current.pt`, report → `outputs/reports/temporal_current.json`.
- **Feature dim:** 283 = 27 coarse (11 base + 16 hand-tracking) + 256 hash. Expect `feature dim 283` after tensor build. Base includes `going_first` from CABT `firstPlayer` (opponent deck rank is **not** a feature).
- **Mac / local:** loads existing rollout JSONL and trains with Torch (CUDA, MPS, or CPU).
- **Linux / Kaggle:** can optionally generate a few CABT rollouts inline when `cg-lib` is available.
- **Does not submit** to the competition leaderboard.

**After pulling code changes:** restart the kernel and run all cells from the top.

Docs: `docs/ARCHITECTURE.md`, `docs/poke-agent-modules.md` (see `game_tracker.py`).

## 1. Setup

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import torch

# Ensure repo root is importable before loading poke_agent.
ROOT = Path.cwd()
if not (ROOT / "requirements.txt").exists() and (ROOT.parent / "requirements.txt").exists():
    ROOT = ROOT.parent.resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from poke_agent.paths import print_runtime_info

print_runtime_info(ROOT)
print("torch", torch.__version__)

## 2. Configuration

Primary settings live in **`poke_agent/config.py`** — edit that file for persistent changes.

This cell builds runtime `CONFIG`. Optional `os.environ[...]` overrides below apply only to this session (see `docs/ARCHITECTURE.md`).

In [ ]:
# Training requires CABT evaluation rollouts from the cg.game engine by default.
os.environ.setdefault("REQUIRE_CABT_EVAL_DATA", "1")

# Optional overrides — uncomment to change defaults for this session.
# os.environ["PRIMARY_ROLLOUT_DATA"] = "data/mac-rollouts-100k-fullstate.jsonl"
# os.environ["MODEL_OUTPUT_PATH"] = "outputs/checkpoints/temporal_current.pt"
# os.environ["DATASET_GAMES"] = "5000"   # 5k games to generate and/or cap training
# os.environ["DATASET_GAMES"] = "100000"  # 100k games
# os.environ["BATCH_GAMES"] = "4"  # games per training batch (lower if VRAM is tight)
# os.environ["REQUIRE_CABT_EVAL_DATA"] = "0"  # smoke test only — allows synthetic fallback

from poke_agent.config import build_config
from poke_agent.features import COARSE_BASE_DIM, COARSE_FEATURE_DIM, DERIVED_INFERENCE_DIM
from poke_agent.game_tracker import DERIVED_FEATURE_NAMES

CONFIG = build_config(ROOT)

print("model_id", CONFIG["model_id"])
print("checkpoint", CONFIG["output_path"])
print("report", CONFIG["report_path"])
print("coarse features", COARSE_FEATURE_DIM, f"({COARSE_BASE_DIM} base + {DERIVED_INFERENCE_DIM} hand-tracking)")
print("expected input dim", COARSE_FEATURE_DIM + CONFIG["state_hash_dim"])
print("derived feature names", list(DERIVED_FEATURE_NAMES))
print("output layout")
for name, path in CONFIG["output_layout"].items():
    print(f"  {name}: {path}")

CONFIG

## 3. Device and simulator

In [ ]:
from poke_agent.device import torch_device
from poke_agent.simulator import load_simulator, print_simulator_status

DEVICE = torch_device()
print("device", DEVICE)

SIMULATOR = load_simulator(ROOT)
print_simulator_status(SIMULATOR)

## 4. Submission deck + optional multi-deck rollout generation

`AGENT_DECK_PATH` is the deck we **submit** to Kaggle (hard-played at runtime). Training is **deck-agnostic** and uses merged multi-deck JSONL.

**Data pipeline (recommended):**
1. `bash scripts/download-episodes-index.sh`
2. `python scripts/scrape_ladder_replays.py --top-percent 1.0`
3. `python scripts/replays_to_rollouts.py --top-percent 1.0 --out data/scraped_rollouts.jsonl`
4. `python scripts/generate_cabt_data.py --episodes 100 --matchups weighted --out data/multideck_rollouts.jsonl`
5. `python scripts/merge_rollouts.py data/scraped_rollouts.jsonl data/multideck_rollouts.jsonl --out data/training_rollouts_merged.jsonl`

Only **complete decisive games** are kept (no truncations/draws/timeouts). Timeout labels use `VALUE_TIMEOUT=-2.0` when present in source data.

Optional inline generation below writes share-weighted archetype matchups to `CONFIG["multideck_rollout_path"]` when the simulator is available.

In [ ]:
from poke_agent.config import resolve_generate_games
from poke_agent.deck import read_deck

DECK, DECK_SOURCE = read_deck(CONFIG, ROOT)
print("submission deck cards", len(DECK))
print("submission deck source", DECK_SOURCE)
print("merged training data", CONFIG["merged_rollout_path"])
print("training sources", [str(path) for path in CONFIG["training_rollout_sources"]])

GENERATE_GAMES = resolve_generate_games(CONFIG)
if GENERATE_GAMES > 0 and SIMULATOR.available:
    import subprocess

    out_path = CONFIG["multideck_rollout_path"]
    print("generating", GENERATE_GAMES, "complete multi-deck games ->", out_path)
    subprocess.run(
        [
            "python",
            "scripts/generate_cabt_data.py",
            "--episodes",
            str(GENERATE_GAMES),
            "--matchups",
            "weighted",
            "--out",
            str(out_path),
        ],
        check=False,
        cwd=ROOT,
    )
else:
    print("skip inline generation; use merged JSONL from scripts/merge_rollouts.py")

## 5. Load dataset and build tensors

Training prefers `CONFIG["merged_rollout_path"]` (deck-agnostic merged JSONL), then scraped/multi-deck sources, then fallbacks.

`GameEventTracker` walks each game sequentially: one tracker and temporal window per game (no cross-game memory).

When `REQUIRE_COMPLETE_GAMES=1` (default), truncated/draw/timeout episodes are dropped before tensor build. Timeout rows that remain in source data use harsh `VALUE_TIMEOUT=-2.0` labels.

In [ ]:
from poke_agent.cabt_validation import (
    assert_cabt_evaluation_rows,
    assert_training_rollout_rows,
    resolve_training_data_path,
    uses_generated_training_data,
)
from poke_agent.dataset import load_jsonl, prepare_training_tensors
from poke_agent.features import COARSE_FEATURE_DIM

EXPECTED_INPUT_DIM = COARSE_FEATURE_DIM + CONFIG["state_hash_dim"]

DATA_PATH = resolve_training_data_path(CONFIG)
if DATA_PATH is None:
    raise RuntimeError(
        "No rollout JSONL found. "
        "Build data/training_rollouts_merged.jsonl via scripts/merge_rollouts.py, "
        "or run section 4 multi-deck generation."
    )

PREVIEW_ROWS = load_jsonl(DATA_PATH)[:5]
if CONFIG.get("require_cabt_eval_data") and not uses_generated_training_data(CONFIG, DATA_PATH):
    assert_cabt_evaluation_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
else:
    assert_training_rollout_rows(PREVIEW_ROWS, path=DATA_PATH, min_rows=1)
print("using games from", DATA_PATH)

TENSORS = prepare_training_tensors(CONFIG, DEVICE)
print("feature dim", TENSORS.x.shape[1], "(expected", EXPECTED_INPUT_DIM, "with hand tracking)")
print("games", TENSORS.num_games, "steps", TENSORS.x.shape[0], "window", TENSORS.window_size)
if TENSORS.x.shape[1] != EXPECTED_INPUT_DIM:
    raise RuntimeError(
        f"unexpected feature dim {TENSORS.x.shape[1]} (expected {EXPECTED_INPUT_DIM}). "
        "Re-run section 4 to regenerate rollouts with full observations."
    )

## 6. Build model and estimate VRAM

Builds the transformer and prints an estimated GPU memory budget before training starts.

In [ ]:
from poke_agent.memory import print_vram_estimate
from poke_agent.training import build_model

MODEL = build_model(CONFIG, TENSORS, DEVICE)
print_vram_estimate(
    model=MODEL,
    param_count=sum(p.numel() for p in MODEL.parameters()),
    tensors=TENSORS,
    config=CONFIG,
    device=DEVICE,
)

## 7. Train

Uses early stopping on total loss. Best weights are restored before checkpoint export.

In [ ]:
from poke_agent.training import train_model

TRAINING_REPORT = train_model(MODEL, TENSORS, CONFIG, DEVICE)
TRAINING_REPORT

## 8. Save checkpoint and report

Writes `outputs/checkpoints/{model_id}.pt` and JSON training report to `outputs/reports/{model_id}.json`.

In [ ]:
from poke_agent.checkpoint import print_training_report, save_checkpoint

OUTPUT_PATH = CONFIG["output_path"]
REPORT_PATH = CONFIG["report_path"]
TRAINING_REPORT = save_checkpoint(
    model=MODEL,
    tensors=TENSORS,
    config=CONFIG,
    training_report=TRAINING_REPORT,
    output_path=OUTPUT_PATH,
)
print_training_report(TRAINING_REPORT, OUTPUT_PATH)
print("report", REPORT_PATH)

## 9. Inspect checkpoint (optional)

In [ ]:
import torch

checkpoint = torch.load(OUTPUT_PATH, map_location="cpu", weights_only=False)
{
    "model_id": checkpoint.get("model_id"),
    "model_type": checkpoint["model_type"],
    "input_dim": checkpoint["input_dim"],
    "coarse_feature_dim": checkpoint.get("coarse_feature_dim"),
    "policy_dim": checkpoint["policy_dim"],
    "model_config": checkpoint["model_config"],
    "best_total_loss": checkpoint["training_report"]["best_total_loss"],
    "best_epoch": checkpoint["training_report"]["best_epoch"],
    "data_path": checkpoint["data_path"],
    "report_path": checkpoint["training_report"].get("report_path"),
}

## 10. Self-play loop (optional)

AlphaGo-style iteration: **collect games** (your deck vs the **field** of meta decks from `decks/competitive/high_performing`) with **beam search**, **evaluate** vs random on field decks, **retrain**, repeat.

Settings in `poke_agent/config.py` under `SELF_PLAY_*` (especially `SELF_PLAY_FIELD_DECK_DIR`). Or run from the repo root:

```bash
python scripts/run_self_play.py --iterations 3 --games 20
```

Use `--no-train` to smoke-test collection only; `--field-deck-dir decks/competitive/the_rest` for a wider field.

In [ ]:
from poke_agent.kaggle_submit import DEFAULT_SUBMISSION_MESSAGE
from poke_agent.self_play import run_self_play_loop, self_play_settings_from_config

SELF_PLAY_SETTINGS = self_play_settings_from_config(
    CONFIG,
    ROOT,
    agent_name=DECK_SOURCE.stem,
    agent_deck=DECK,
)
# Quick smoke: 1 iteration, 2 games, skip training
# SELF_PLAY_SETTINGS.iterations = 1
# SELF_PLAY_SETTINGS.games_per_iteration = 2
# SELF_PLAY_SETTINGS.train_after_collect = False

SELF_PLAY_REPORTS = run_self_play_loop(
    config=CONFIG,
    simulator=SIMULATOR,
    agent_deck=DECK,
    agent_name=DECK_SOURCE.stem,
    settings=SELF_PLAY_SETTINGS,
    device=DEVICE,
    initial_checkpoint=CONFIG["output_path"],
    submit_on_stop=True,
    submission_message=DEFAULT_SUBMISSION_MESSAGE,
    root=ROOT,
)
SELF_PLAY_REPORTS